# Can we predict 30-day readmission?
Follows `checklist.md`. Plan:
- No date column exists, and `person_key` repeats (several stays per person), so the split is by **whole person** (checklist 1, 2).
- Features are limited to what is known at prediction time (discharge): all columns in `columns.csv` except `person_key` (identifier) and the target. `length_of_stay_days` and `n_medications` (at discharge) are known when the person leaves, which is when a readmission prediction would be made.

In [ ]:
import numpy as np, pandas as pd, json
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, brier_score_loss
df=pd.read_csv('data.csv')
TARGET='readmit_30d'; ID='person_key'
print(df.shape, df[ID].nunique(), 'persons; prevalence', df[TARGET].mean().round(3))
print(df.isna().sum().sum(),'missing values')

## 1. Hold out a test set of whole persons
No person appears in both train and test. The test set is not looked at until step 6.

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=0)
tr_idx,te_idx=next(gss.split(df,df[TARGET],groups=df[ID]))
train,test=df.iloc[tr_idx].reset_index(drop=True),df.iloc[te_idx].reset_index(drop=True)
assert set(train[ID]).isdisjoint(test[ID])
FEATURES=[c for c in df.columns if c not in (ID,TARGET)]
X,y,g=train[FEATURES],train[TARGET],train[ID]
print(len(train),len(test),FEATURES)

## 2-4. Candidate models (named before any scoring) and CV scheme
Candidates, all with preprocessing inside the pipeline so it is refit per fold:
1. `baseline`: predicts training prevalence (trivial).
2. `logreg`: standardised numeric + one-hot sex/postcode, L2 logistic regression, C in {0.01, 0.1, 1}.
3. `hgb`: gradient boosting, learning_rate in {0.03, 0.1}, max_depth in {2, 3}, 200 iterations max with early stopping.

CV: 5-fold `StratifiedGroupKFold` on training persons only. Selection metric: ROC AUC (threshold-free). Threshold is chosen afterwards on out-of-fold predictions of the chosen model.

In [ ]:
num=[c for c in FEATURES if c not in ('sex','postcode_area')]
def prep(scale):
    if scale:
        return ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),['sex','postcode_area'])])
    return ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),['sex']),('z','passthrough',['postcode_area'])])
candidates={'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))])}
for C_ in (0.01,0.1,1):
    candidates['logreg_C%s'%C_]=Pipeline([('p',prep(True)),('m',LogisticRegression(C=C_,max_iter=2000))])
for lr in (0.03,0.1):
    for d in (2,3):
        candidates['hgb_lr%s_d%s'%(lr,d)]=Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(learning_rate=lr,max_depth=d,max_iter=200,early_stopping=True,random_state=0))])
print(list(candidates))

In [ ]:
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=1)
folds=list(cv.split(X,y,groups=g))
oof={};rows=[]
for name,pipe in candidates.items():
    p=np.zeros(len(train));aucs=[]
    for a,b in folds:
        m=clone(pipe).fit(X.iloc[a],y.iloc[a]); p[b]=m.predict_proba(X.iloc[b])[:,1]
        aucs.append(roc_auc_score(y.iloc[b],p[b]))
    oof[name]=p; rows.append((name,np.mean(aucs),np.std(aucs),average_precision_score(y,p)))
cvres=pd.DataFrame(rows,columns=['model','cv_auc','auc_sd','oof_pr_auc']).sort_values('cv_auc',ascending=False)
cvres

## 5. Choose model and threshold on CV only, and record the choice

In [ ]:
best=cvres.iloc[0].model
if best=='baseline': print('No candidate beats the baseline.')
p=oof[best]
ths=np.linspace(0.05,0.8,76)
f1s=[f1_score(y,p>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
choice={'model':best,'threshold':thr,'cv_auc':float(cvres.iloc[0].cv_auc),'cv_f1_at_threshold':float(max(f1s))}
json.dump(choice,open('chosen_model.json','w'))
print(choice)
print('OOF precision/recall at threshold:',round(precision_score(y,p>=thr),3),round(recall_score(y,p>=thr),3))

**Recorded choice (printed above and saved to `chosen_model.json`, fixed before the test set is scored):** the top CV-AUC candidate, with the F1-maximising threshold on out-of-fold predictions. Nothing below changes it.

## 6. Refit on all training rows, score the test set once

In [ ]:
final=clone(candidates[choice['model']]).fit(X,y)
pt=final.predict_proba(test[FEATURES])[:,1]; yt=test[TARGET]
rng=np.random.default_rng(0); ids=test[ID].unique(); grp={k:v.index.values for k,v in test.groupby(ID)}
bs=[]
for _ in range(500):  # person-level bootstrap CI
    s=np.concatenate([grp[k] for k in rng.choice(ids,len(ids))]); bs.append(roc_auc_score(yt.iloc[s],pt[s]))
yh=pt>=choice['threshold']
print('TEST ROC AUC %.3f (95%% CI %.3f-%.3f)'%(roc_auc_score(yt,pt),np.percentile(bs,2.5),np.percentile(bs,97.5)))
print('PR AUC %.3f vs prevalence %.3f'%(average_precision_score(yt,pt),yt.mean()))
print('at threshold %.2f: precision %.3f recall %.3f F1 %.3f'%(choice['threshold'],precision_score(yt,yh),recall_score(yt,yh),f1_score(yt,yh)))
print('Brier %.3f'%brier_score_loss(yt,pt))

## Conclusion
See the printed test results above (the headline is the single test-set score). Interpret against the baseline (AUC 0.5, PR AUC = prevalence). Caveats: no calendar date was available, so the split is by person rather than by time; if the model will be used on future admissions, temporal drift is untested.